# Phase 6 - Variational Quantum Classifier: ZZ Feature Map

A **variational quantum classifier (VQC)** has three parts:

1. A **feature map** that encodes the data into a quantum state (this is the
   part that changes between notebooks - here it is **zz feature map**).
2. A trainable **ansatz**, a circuit with adjustable rotation parameters.
3. A classical **optimizer** that tunes those parameters to reduce
   classification error.

The ZZ feature map adds **entangling gates** on top of single-qubit rotations, encoding interactions between features. This is the encoding our hypothesis predicts will behave measurably differently from angle encoding.

We reuse the exact dataset, split, and scaling from Phase 2 so the result is
directly comparable to the classical baseline and to the other encodings.

In [ ]:
"""06_vqc_zz_feature_map.ipynb"""

# Cell 01 - Imports and reproducible seed

import qml_utils as hu
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit.circuit.library import real_amplitudes, zz_feature_map
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.utils import algorithm_globals

algorithm_globals.random_seed = hu.RANDOM_SEED
np.random.seed(hu.RANDOM_SEED)

---
**Cell 02 - Load and scale the data (same as Phase 2).**

In [ ]:
# Cell 02 - Prepare the two-moons data, scaled to [0, pi]

x_train, x_test, y_train, y_test = hu.load_moons(n_samples=200, noise=0.20)
x_train_scaled, x_test_scaled, scaler = hu.scale_features(
    x_train, x_test, feature_range=(0.0, np.pi)
)
print(f"Train: {x_train_scaled.shape}, Test: {x_test_scaled.shape}")

---
**Cell 03 - Build the encoding and the ansatz.**
The feature map is `zz_feature_map`. Decomposing it reveals the entangling CX
gates that distinguish it from angle encoding. The ansatz is again
`real_amplitudes`, so only the encoding has changed.

In [ ]:
# Cell 03 - Feature map + ansatz

feature_map = zz_feature_map(feature_dimension=2, reps=2)
ansatz = real_amplitudes(num_qubits=2, reps=2)

full_circuit = feature_map.compose(ansatz)
print(f"Number of qubits        : {full_circuit.num_qubits}")
print(f"Trainable parameters    : {ansatz.num_parameters}")
print(f"Circuit depth (composed): {full_circuit.decompose().depth()}")
display(full_circuit.decompose().draw(output="mpl"))

---
**Cell 04 - Train the classifier.**
Same training recipe as the other VQCs.

In [ ]:
# Cell 04 - Train the ZZ-feature-map VQC

objective_values = []


def store_objective(weights, value):
    objective_values.append(value)


vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=80),
    callback=store_objective,
)

vqc.fit(x_train_scaled, y_train)

zz_train_acc = vqc.score(x_train_scaled, y_train)
zz_test_acc = vqc.score(x_test_scaled, y_test)
print(f"ZZ-feature-map VQC training accuracy: {zz_train_acc:.3f}")
print(f"ZZ-feature-map VQC test accuracy    : {zz_test_acc:.3f}")

---
**Cell 05 - Training convergence.**

In [ ]:
# Cell 05 - Plot the optimization curve

plt.figure(figsize=(7, 4))
plt.plot(objective_values)
plt.xlabel("optimizer iteration")
plt.ylabel("objective value")
plt.title("ZZ-feature-map VQC training convergence")
plt.grid(True, alpha=0.3)
plt.show()

---
**Cell 06 - Decision boundary.**
This is the key comparison for the hypothesis. Does the entangling ZZ feature
map carve out a boundary that looks different from the angle-encoding boundary
in Phase 4?

In [ ]:
# Cell 06 - Plot the learned decision boundary

hu.plot_decision_boundary(
    vqc.predict,
    x_test_scaled,
    y_test,
    title=f"ZZ-feature-map VQC (test accuracy {zz_test_acc:.2f})",
    grid_steps=40,
)
plt.show()

---
## Interpretation

Compare this decision boundary directly with the angle-encoding boundary from
Phase 4. The hypothesis predicts they will be **measurably different** because
of entanglement, even if neither beats the classical SVM. Record the **test
accuracy** for the final comparison in Phase 7.